# Market Insights

## Nguồn dữ liệu
Dữ liệu được trích xuất và tính toán trước từ kiến trúc Lakehouse lưu trữ tại schema `lakehouse.mart`:
- `mart_macro_daily`: Chuỗi thời gian hàng ngày về các chỉ báo vĩ mô (lãi suất điều hành Fed, lợi suất trái phiếu 2Y và 10Y, độ dốc 10Y-2Y, VIX, High Yield spread, S&P 500).
- `mart_concentration`: Tỷ trọng vốn hoá tích lũy của Top 10, Top 50, Top 500 cổ phiếu vào ngày cuối mỗi tháng.
- `mart_size_growth`: Giá trị tích luỹ danh mục ($1 đầu tư ban đầu) phân loại theo quy mô vốn hoá (Large, Mid, Small Cap).
- `mart_style_growth`: Giá trị tích luỹ danh mục ($1 đầu tư ban đầu) phân loại theo trường phái định giá P/E (Value, Core, Growth, Loss-making).
- `mart_sector_year_return`: Lợi suất bình quân hàng năm (2017–2025) của 11 ngành kinh tế.
- `mart_sector_latest`: Dữ liệu tổng hợp mới nhất của 11 ngành (số lượng mã, vốn hoá, P/E trung vị, ROE trung vị, biên lợi nhuận ròng trung vị).

In [1]:
import warnings
warnings.filterwarnings('ignore')

import trino
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display, Markdown

# Cấu hình renderer cho Plotly để xuất mime type plotly cho nbconvert
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Kết nối Trino Lakehouse
conn = trino.dbapi.connect(
    host='localhost',
    port=8185,
    user='alpha',
    catalog='lakehouse',
    schema='mart'
)

def load_table(table_name):
    cur = conn.cursor()
    cur.execute(f"SELECT * FROM lakehouse.mart.{table_name}")
    cols = [c[0] for c in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

# Tải dữ liệu từ 6 bảng mart
df_macro = load_table("mart_macro_daily")
df_macro['date'] = pd.to_datetime(df_macro['date'])
for c in ['fed_funds_rate', 'yield_2y', 'yield_10y', 'curve_10y_2y', 'vix', 'credit_spread_hy', 'sp500_index']:
    df_macro[c] = pd.to_numeric(df_macro[c], errors='coerce')
df_macro = df_macro.sort_values('date').reset_index(drop=True)

df_conc = load_table("mart_concentration")
df_conc['date'] = pd.to_datetime(df_conc['date'])
for c in ['top10_share_pct', 'top50_share_pct', 'top500_share_pct']:
    df_conc[c] = pd.to_numeric(df_conc[c], errors='coerce')
df_conc = df_conc.sort_values('date').reset_index(drop=True)

df_size = load_table("mart_size_growth")
df_size['date'] = pd.to_datetime(df_size['date'])
df_size['growth_of_1'] = pd.to_numeric(df_size['growth_of_1'], errors='coerce')
df_size = df_size.sort_values(['bucket', 'date']).reset_index(drop=True)

df_style = load_table("mart_style_growth")
df_style['date'] = pd.to_datetime(df_style['date'])
df_style['growth_of_1'] = pd.to_numeric(df_style['growth_of_1'], errors='coerce')
df_style = df_style.sort_values(['bucket', 'date']).reset_index(drop=True)

df_sec_yr = load_table("mart_sector_year_return")
df_sec_yr['return_pct'] = pd.to_numeric(df_sec_yr['return_pct'], errors='coerce')
df_sec_yr = df_sec_yr.sort_values(['year', 'sector']).reset_index(drop=True)

df_sec_lat = load_table("mart_sector_latest")
for c in ['n_stocks', 'total_mcap_bn', 'median_pe', 'median_roe_pct', 'median_net_margin_pct']:
    df_sec_lat[c] = pd.to_numeric(df_sec_lat[c], errors='coerce')


In [2]:
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=df_macro['date'],
    y=df_macro['fed_funds_rate'],
    name='Lãi suất Fed (Fed Funds Rate)',
    line=dict(color='#1f77b4', width=2)
))
fig1.add_trace(go.Scatter(
    x=df_macro['date'],
    y=df_macro['curve_10y_2y'],
    name='Độ dốc đường cong 10Y-2Y (Yield Curve)',
    line=dict(color='#d62728', width=2)
))
fig1.add_hline(
    y=0,
    line_dash='dash',
    line_color='#7f7f7f',
    annotation_text='Ngưỡng 0 (Đảo chiều khi âm)',
    annotation_position='bottom right'
)
fig1.update_layout(
    title='Biểu đồ 1: Lãi suất Fed và Độ dốc Đường cong Lợi suất 10Y-2Y (2016–2026)',
    xaxis_title='Thời gian',
    yaxis_title='Lãi suất / Chênh lệch (%)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig1.show()


In [3]:
min_spread_row = df_macro.loc[df_macro['curve_10y_2y'].idxmin()]
min_spread_date = min_spread_row['date'].strftime('%Y-%m-%d')
min_spread_val = float(min_spread_row['curve_10y_2y'])
neg_spread_days = int((df_macro['curve_10y_2y'] < 0).sum())
max_fed_val = float(df_macro['fed_funds_rate'].max())

display(Markdown(f"""### Nhận định: Lãi suất Fed và Đường cong Lợi suất 10Y-2Y
- **Hiện tượng đảo chiều đường cong:** Đường cong lợi suất 10Y-2Y chạm mức âm sâu nhất là **{min_spread_val:.2f}%** vào ngày **{min_spread_date}**.
- **Kéo dài giai đoạn nghịch đảo:** Tổng số ngày lịch ghi nhận đường cong lợi suất âm (spread < 0) lên tới **{neg_spread_days} ngày lịch**, cho thấy thị trường duy trì kỳ vọng kéo dài về rủi ro suy thoái kinh tế.
- **Sự đi cùng chính sách thắt chặt:** Chu kỳ tăng lãi suất của Fed lên đỉnh **{max_fed_val:.2f}%** đi cùng với sự thu hẹp biên độ lợi suất kỳ hạn dài so với kỳ hạn ngắn và hiện tượng đảo chiều đường cong lợi suất."""))


### Nhận định: Lãi suất Fed và Đường cong Lợi suất 10Y-2Y
- **Hiện tượng đảo chiều đường cong:** Đường cong lợi suất 10Y-2Y chạm mức âm sâu nhất là **-1.08%** vào ngày **2023-07-05**.
- **Kéo dài giai đoạn nghịch đảo:** Tổng số ngày lịch ghi nhận đường cong lợi suất âm (spread < 0) lên tới **795 ngày lịch**, cho thấy thị trường duy trì kỳ vọng kéo dài về rủi ro suy thoái kinh tế.
- **Sự đi cùng chính sách thắt chặt:** Chu kỳ tăng lãi suất của Fed lên đỉnh **5.33%** đi cùng với sự thu hẹp biên độ lợi suất kỳ hạn dài so với kỳ hạn ngắn và hiện tượng đảo chiều đường cong lợi suất.

In [4]:
fig2 = make_subplots(specs=[[{"secondary_y": True}]])
fig2.add_trace(
    go.Scatter(
        x=df_macro['date'],
        y=df_macro['vix'],
        name='Chỉ số Biến động VIX',
        line=dict(color='#ff7f0e', width=1.8)
    ),
    secondary_y=False
)
df_hy_valid = df_macro.dropna(subset=['credit_spread_hy'])
fig2.add_trace(
    go.Scatter(
        x=df_hy_valid['date'],
        y=df_hy_valid['credit_spread_hy'],
        name='Chênh lệch Tín dụng HY (Credit Spread HY)',
        line=dict(color='#2ca02c', width=2)
    ),
    secondary_y=True
)
fig2.update_layout(
    title='Chỉ số Biến động VIX và Chênh lệch Tín dụng High Yield',
    xaxis_title='Thời gian',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig2.update_yaxes(title_text='Chỉ số VIX (điểm)', secondary_y=False)
fig2.update_yaxes(title_text='HY Spread (%)', secondary_y=True)
fig2.show()


In [5]:
max_vix_row = df_macro.loc[df_macro['vix'].idxmax()]
max_vix_date = max_vix_row['date'].strftime('%Y-%m-%d')
max_vix_val = float(max_vix_row['vix'])

df_hy_nonnull = df_macro.dropna(subset=['credit_spread_hy'])
hy_start_date = df_hy_nonnull['date'].min().strftime('%Y-%m-%d')
hy_days_count = len(df_hy_nonnull)
max_hy_row = df_hy_nonnull.loc[df_hy_nonnull['credit_spread_hy'].idxmax()]
max_hy_date = max_hy_row['date'].strftime('%Y-%m-%d')
max_hy_val = float(max_hy_row['credit_spread_hy'])
min_hy_val = float(df_hy_nonnull['credit_spread_hy'].min())

display(Markdown(f"""### Nhận định: VIX và Chênh lệch Tín dụng High Yield
- **Đỉnh điểm biến động cổ phiếu:** Chỉ số VIX ghi nhận mức cao kỷ lục **{max_vix_val:.2f} điểm** vào ngày **{max_vix_date}** (ngày trong bảng macro có thể lệch ~1 ngày so với ngày quan sát (chưa kiểm chứng nguyên nhân; có thể là ngày dữ liệu khả dụng)).
- **Dữ liệu chênh lệch tín dụng High Yield:** Chuỗi dữ liệu credit spread HY bắt đầu từ ngày **{hy_start_date}** với tổng cộng **{hy_days_count} ngày lịch** có dữ liệu (trước thời điểm này là giá trị NULL).
- **Biên độ rủi ro tín dụng doanh nghiệp:** Trong giai đoạn có dữ liệu theo dõi, chênh lệch tín dụng High Yield dao động trong khoảng từ **{min_hy_val:.2f}%** đến mức đỉnh **{max_hy_val:.2f}%** vào ngày **{max_hy_date}**."""))


### Nhận định: VIX và Chênh lệch Tín dụng High Yield
- **Đỉnh điểm biến động cổ phiếu:** Chỉ số VIX ghi nhận mức cao kỷ lục **82.69 điểm** vào ngày **2020-03-17** (ngày trong bảng macro có thể lệch ~1 ngày so với ngày quan sát (chưa kiểm chứng nguyên nhân; có thể là ngày dữ liệu khả dụng)).
- **Dữ liệu chênh lệch tín dụng High Yield:** Chuỗi dữ liệu credit spread HY bắt đầu từ ngày **2023-10-17** với tổng cộng **1095 ngày lịch** có dữ liệu (trước thời điểm này là giá trị NULL).
- **Biên độ rủi ro tín dụng doanh nghiệp:** Trong giai đoạn có dữ liệu theo dõi, chênh lệch tín dụng High Yield dao động trong khoảng từ **2.59%** đến mức đỉnh **4.61%** vào ngày **2025-04-08**.

In [6]:
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=df_conc['date'],
    y=df_conc['top10_share_pct'],
    name='Top 10 Cổ phiếu',
    line=dict(color='#1f77b4', width=2.2)
))
fig3.add_trace(go.Scatter(
    x=df_conc['date'],
    y=df_conc['top50_share_pct'],
    name='Top 50 Cổ phiếu',
    line=dict(color='#ff7f0e', width=2.2)
))
fig3.add_trace(go.Scatter(
    x=df_conc['date'],
    y=df_conc['top500_share_pct'],
    name='Top 500 Cổ phiếu',
    line=dict(color='#2ca02c', width=2.2)
))
fig3.update_layout(
    title='Mức độ Tập trung Vốn hoá Thị trường (Top 10, Top 50, Top 500)',
    xaxis_title='Thời gian',
    yaxis_title='Tỷ trọng Vốn hoá (%)',
    yaxis=dict(range=[0, 100]),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig3.show()


In [7]:
start_date_c = df_conc.iloc[0]['date'].strftime('%Y-%m-%d')
end_date_c = df_conc.iloc[-1]['date'].strftime('%Y-%m-%d')
start_t10 = float(df_conc.iloc[0]['top10_share_pct'])
end_t10 = float(df_conc.iloc[-1]['top10_share_pct'])
start_t50 = float(df_conc.iloc[0]['top50_share_pct'])
end_t50 = float(df_conc.iloc[-1]['top50_share_pct'])
end_t500 = float(df_conc.iloc[-1]['top500_share_pct'])
t10_mult = end_t10 / start_t10

display(Markdown(f"""### Nhận định: Xu hướng Tập trung Vốn hoá Thị trường
- **Sự bùng nổ tỷ trọng Top 10:** Tỷ trọng vốn hoá của Top 10 cổ phiếu lớn nhất đã tăng từ **{start_t10:.2f}%** vào ngày **{start_date_c}** lên tới **{end_t10:.2f}%** vào ngày **{end_date_c}** (tăng gấp **{t10_mult:.2f} lần**).
- **Mức độ chi phối của nhóm dẫn đầu:** Top 50 mở rộng thị phần từ **{start_t50:.2f}%** lên **{end_t50:.2f}%**, trong khi Top 500 nắm giữ áp đảo **{end_t500:.2f}%** toàn bộ giá trị vốn hoá thị trường tại thời điểm kết thúc kỳ quan sát.
- **Góc nhìn cấu trúc:** Dữ liệu cho thấy tỷ trọng vốn hoá thị trường tập trung ngày càng dày đặc vào nhóm cổ phiếu quy mô lớn nhất (giả thuyết về sự ưu tiên của dòng vốn thụ động đối với các mã dẫn đầu cần kiểm chứng thêm qua dữ liệu chi tiết)."""))


### Nhận định: Xu hướng Tập trung Vốn hoá Thị trường
- **Sự bùng nổ tỷ trọng Top 10:** Tỷ trọng vốn hoá của Top 10 cổ phiếu lớn nhất đã tăng từ **12.13%** vào ngày **2016-09-30** lên tới **32.81%** vào ngày **2026-10-15** (tăng gấp **2.70 lần**).
- **Mức độ chi phối của nhóm dẫn đầu:** Top 50 mở rộng thị phần từ **32.78%** lên **54.03%**, trong khi Top 500 nắm giữ áp đảo **87.86%** toàn bộ giá trị vốn hoá thị trường tại thời điểm kết thúc kỳ quan sát.
- **Góc nhìn cấu trúc:** Dữ liệu cho thấy tỷ trọng vốn hoá thị trường tập trung ngày càng dày đặc vào nhóm cổ phiếu quy mô lớn nhất (giả thuyết về sự ưu tiên của dòng vốn thụ động đối với các mã dẫn đầu cần kiểm chứng thêm qua dữ liệu chi tiết).

In [8]:
fig4 = go.Figure()
colors_size = {'1 Large (>10B)': '#1f77b4', '2 Mid (2-10B)': '#ff7f0e', '3 Small (0.3-2B)': '#2ca02c'}
for bucket in sorted(df_size['bucket'].unique()):
    df_b = df_size[df_size['bucket'] == bucket]
    fig4.add_trace(go.Scatter(
        x=df_b['date'],
        y=df_b['growth_of_1'],
        name=bucket,
        line=dict(color=colors_size.get(bucket, '#333333'), width=2)
    ))
fig4.update_layout(
    title='Biểu đồ 4: Tăng trưởng Lũy kế của 1 USD theo Quy mô Vốn hoá (Equal-Weight)',
    xaxis_title='Thời gian',
    yaxis_title='Giá trị Danh mục (USD)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig4.show()


In [9]:
last_dt_size = df_size['date'].max()
last_size_snap = df_size[df_size['date'] == last_dt_size].sort_values('growth_of_1', ascending=False)
win_size_row = last_size_snap.iloc[0]
lose_size_row = last_size_snap.iloc[-1]
mid_size_row = last_size_snap.iloc[1]

win_size_name = win_size_row['bucket']
win_size_val = float(win_size_row['growth_of_1'])
mid_size_name = mid_size_row['bucket']
mid_size_val = float(mid_size_row['growth_of_1'])
lose_size_name = lose_size_row['bucket']
lose_size_val = float(lose_size_row['growth_of_1'])

display(Markdown(f"""### Nhận định: Hiệu suất Tăng trưởng theo Quy mô Vốn hoá
- **Nhóm dẫn đầu cuối kỳ:** Nhóm cổ phiếu quy mô vốn hoá lớn (**{win_size_name}**) ghi nhận giá trị tích luỹ cao nhất, đạt **{win_size_val:.4f} USD** từ 1 USD ban đầu.
- **Thứ hạng các nhóm quy mô:** Nhóm **{mid_size_name}** đạt vị trí thứ hai với **{mid_size_val:.4f} USD**, trong khi nhóm có giá trị tích luỹ thấp nhất là **{lose_size_name}** đạt **{lose_size_val:.4f} USD**.
- **Tương quan quy mô:** Kết quả cho thấy trong giai đoạn khảo sát, mức tăng trưởng giá trị danh mục đi cùng với thứ bậc quy mô vốn hoá (giả thuyết về sự khác biệt dòng vốn giữa các nhóm quy mô cần kiểm chứng thêm qua dữ liệu chi tiết)."""))


### Nhận định: Hiệu suất Tăng trưởng theo Quy mô Vốn hoá
- **Nhóm dẫn đầu cuối kỳ:** Nhóm cổ phiếu quy mô vốn hoá lớn (**1 Large (>10B)**) ghi nhận giá trị tích luỹ cao nhất, đạt **2.7881 USD** từ 1 USD ban đầu.
- **Thứ hạng các nhóm quy mô:** Nhóm **2 Mid (2-10B)** đạt vị trí thứ hai với **2.4250 USD**, trong khi nhóm có giá trị tích luỹ thấp nhất là **3 Small (0.3-2B)** đạt **2.0886 USD**.
- **Tương quan quy mô:** Kết quả cho thấy trong giai đoạn khảo sát, mức tăng trưởng giá trị danh mục đi cùng với thứ bậc quy mô vốn hoá (giả thuyết về sự khác biệt dòng vốn giữa các nhóm quy mô cần kiểm chứng thêm qua dữ liệu chi tiết).

In [10]:
fig5 = go.Figure()
colors_style = {
    '1 Value (P/E<12)': '#2ca02c',
    '2 Core (12-30)': '#1f77b4',
    '3 Growth (P/E>=30)': '#ff7f0e',
    '4 Loss-making (P/E n/a)': '#d62728'
}
for bucket in sorted(df_style['bucket'].unique()):
    df_b = df_style[df_style['bucket'] == bucket]
    fig5.add_trace(go.Scatter(
        x=df_b['date'],
        y=df_b['growth_of_1'],
        name=bucket,
        line=dict(color=colors_style.get(bucket, '#333333'), width=2)
    ))
fig5.update_layout(
    title='Tăng trưởng Lũy kế của 1 USD theo Phong cách Định giá (Value vs Growth)',
    xaxis_title='Thời gian',
    yaxis_title='Giá trị Danh mục (USD)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig5.show()


In [11]:
last_dt_style = df_style['date'].max()
last_style_snap = df_style[df_style['date'] == last_dt_style].sort_values('growth_of_1', ascending=False)
win_style_row = last_style_snap.iloc[0]
lose_style_row = last_style_snap.iloc[-1]

win_style_name = win_style_row['bucket']
win_style_val = float(win_style_row['growth_of_1'])
lose_style_name = lose_style_row['bucket']
lose_style_val = float(lose_style_row['growth_of_1'])
core_val = float(last_style_snap[last_style_snap['bucket'].str.contains('Core')]['growth_of_1'].iloc[0])
growth_val = float(last_style_snap[last_style_snap['bucket'].str.contains('Growth')]['growth_of_1'].iloc[0])

display(Markdown(f"""### Nhận định: Phong cách Đầu tư (Value vs Growth)
- **Chiến lược giá trị dẫn đầu:** Nhóm **{win_style_name}** ghi nhận hiệu suất tích lũy cao nhất cuối chu kỳ, đạt **{win_style_val:.4f} USD** từ 1 USD đầu tư, xếp trên nhóm Core (**{core_val:.4f} USD**) và Growth (**{growth_val:.4f} USD**).
- **Hiệu suất nhóm doanh nghiệp thua lỗ:** Nhóm **{lose_style_name}** ghi nhận giá trị tích luỹ thấp nhất với **{lose_style_val:.4f} USD**, cho thấy sự phân hóa mạnh mẽ từ thị trường.
- **Đặc thù danh mục Equal-Weight:** Khi được phân bổ theo tỷ trọng đều và tái cân bằng định kỳ, danh mục cổ phiếu định giá thấp (Value) đi cùng với hiệu suất sinh lời cao hơn các nhóm còn lại trong khung thời gian khảo sát (giả thuyết về phần bù rủi ro giá trị cần kiểm chứng thêm)."""))


### Nhận định: Phong cách Đầu tư (Value vs Growth)
- **Chiến lược giá trị dẫn đầu:** Nhóm **1 Value (P/E<12)** ghi nhận hiệu suất tích lũy cao nhất cuối chu kỳ, đạt **2.9165 USD** từ 1 USD đầu tư, xếp trên nhóm Core (**2.5931 USD**) và Growth (**2.4035 USD**).
- **Hiệu suất nhóm doanh nghiệp thua lỗ:** Nhóm **4 Loss-making (P/E n/a)** ghi nhận giá trị tích luỹ thấp nhất với **1.7038 USD**, cho thấy sự phân hóa mạnh mẽ từ thị trường.
- **Đặc thù danh mục Equal-Weight:** Khi được phân bổ theo tỷ trọng đều và tái cân bằng định kỳ, danh mục cổ phiếu định giá thấp (Value) đi cùng với hiệu suất sinh lời cao hơn các nhóm còn lại trong khung thời gian khảo sát (giả thuyết về phần bù rủi ro giá trị cần kiểm chứng thêm).

In [12]:
pivot_return = df_sec_yr.pivot(index='sector', columns='year', values='return_pct')
fig6 = px.imshow(
    pivot_return,
    labels=dict(x="Năm", y="Ngành", color="Lợi suất (%)"),
    x=pivot_return.columns,
    y=pivot_return.index,
    color_continuous_scale='RdYlGn',
    text_auto='.1f',
    aspect='auto'
)
fig6.update_layout(
    title='Heatmap Lợi suất Bình quân Hàng năm theo Ngành (2017–2025)',
    xaxis_title='Năm',
    yaxis_title='Ngành Kinh tế',
    template='plotly_white'
)
fig6.show()


In [13]:
best_per_yr = df_sec_yr.sort_values(['year', 'return_pct'], ascending=[True, False]).groupby('year').first().reset_index()
best_str = ", ".join([f"**{r['year']}**: {r['sector']} (+{float(r['return_pct']):.1f}%)" for _, r in best_per_yr.iterrows()])

max_ret_row = df_sec_yr.loc[df_sec_yr['return_pct'].idxmax()]
min_ret_row = df_sec_yr.loc[df_sec_yr['return_pct'].idxmin()]

display(Markdown(f"""### Nhận định: Luân chuyển Hiệu suất theo Ngành (2017–2025)
- **Ngành dẫn đầu qua các năm:** {best_str}.
- **Kỷ lục tăng trưởng đơn năm:** Mức sinh lời cao nhất ghi nhận được thuộc về ngành **{max_ret_row['sector']}** vào năm **{max_ret_row['year']}** với mức tăng **+{float(max_ret_row['return_pct']):.1f}%**.
- **Mức giảm sâu nhất trong một năm:** Thuộc về ngành **{min_ret_row['sector']}** vào năm **{min_ret_row['year']}** với mức giảm **{float(min_ret_row['return_pct']):.1f}%**.
- **Tính chu kỳ kinh tế:** Diễn biến lợi suất cho thấy sự biến động và phân hóa mạnh mẽ giữa các ngành qua từng năm, đi cùng với sự luân chuyển vị trí dẫn đầu giữa các nhóm ngành như công nghệ, năng lượng hay y tế."""))


### Nhận định: Luân chuyển Hiệu suất theo Ngành (2017–2025)
- **Ngành dẫn đầu qua các năm:** **2017**: Healthcare (+30.5%), **2018**: Utilities (+1.9%), **2019**: Technology (+38.2%), **2020**: Technology (+53.7%), **2021**: Energy (+54.2%), **2022**: Energy (+50.7%), **2023**: Industrials (+23.8%), **2024**: Financial Services (+25.7%), **2025**: Healthcare (+27.8%).
- **Kỷ lục tăng trưởng đơn năm:** Mức sinh lời cao nhất ghi nhận được thuộc về ngành **Energy** vào năm **2021** với mức tăng **+54.2%**.
- **Mức giảm sâu nhất trong một năm:** Thuộc về ngành **Technology** vào năm **2022** với mức giảm **-34.9%**.
- **Tính chu kỳ kinh tế:** Diễn biến lợi suất cho thấy sự biến động và phân hóa mạnh mẽ giữa các ngành qua từng năm, đi cùng với sự luân chuyển vị trí dẫn đầu giữa các nhóm ngành như công nghệ, năng lượng hay y tế.

In [14]:
df_sec_sorted_pe = df_sec_lat.sort_values('median_pe', ascending=True)
fig7 = go.Figure(go.Bar(
    x=df_sec_sorted_pe['median_pe'],
    y=df_sec_sorted_pe['sector'],
    orientation='h',
    text=df_sec_sorted_pe['median_pe'].apply(lambda x: f'{x:.1f}x'),
    textposition='auto',
    marker=dict(
        color=df_sec_sorted_pe['median_pe'],
        colorscale='Blues'
    )
))
fig7.update_layout(
    title='Định giá P/E Trung vị theo Ngành (Snapshot Mới nhất)',
    xaxis_title='P/E Trung vị (lần)',
    yaxis_title='Ngành Kinh tế',
    template='plotly_white'
)
fig7.show()


In [15]:
max_pe_row = df_sec_lat.loc[df_sec_lat['median_pe'].idxmax()]
min_pe_row = df_sec_lat.loc[df_sec_lat['median_pe'].idxmin()]
max_pe_sector = max_pe_row['sector']
max_pe_val = float(max_pe_row['median_pe'])
min_pe_sector = min_pe_row['sector']
min_pe_val = float(min_pe_row['median_pe'])

display(Markdown(f"""### Nhận định: Định giá P/E Trung vị theo Ngành
- **Ngành có mức định giá cao nhất:** Ngành **{max_pe_sector}** ghi nhận mức định giá P/E trung vị cao nhất đạt **{max_pe_val:.1f}x**.
- **Ngành có định giá thấp nhất:** Ngành **{min_pe_sector}** có mức định giá thấp nhất thị trường với P/E trung vị chỉ **{min_pe_val:.1f}x**.
- **Biên độ phân hóa định giá:** Mức định giá giữa ngành cao nhất và thấp nhất chênh lệch hơn hai lần (giả thuyết về sự khác nhau trong kỳ vọng tăng trưởng đối với từng mô hình kinh doanh cần kiểm chứng thêm)."""))


### Nhận định: Định giá P/E Trung vị theo Ngành
- **Ngành có mức định giá cao nhất:** Ngành **Technology** ghi nhận mức định giá P/E trung vị cao nhất đạt **28.2x**.
- **Ngành có định giá thấp nhất:** Ngành **Financial Services** có mức định giá thấp nhất thị trường với P/E trung vị chỉ **12.3x**.
- **Biên độ phân hóa định giá:** Mức định giá giữa ngành cao nhất và thấp nhất chênh lệch hơn hai lần (giả thuyết về sự khác nhau trong kỳ vọng tăng trưởng đối với từng mô hình kinh doanh cần kiểm chứng thêm).

In [16]:
fig8 = px.scatter(
    df_sec_lat,
    x='median_pe',
    y='median_roe_pct',
    size='total_mcap_bn',
    color='sector',
    hover_name='sector',
    text='sector',
    size_max=50,
    labels=dict(
        median_pe='P/E Trung vị (lần)',
        median_roe_pct='ROE Trung vị (%)',
        total_mcap_bn='Tổng Vốn hoá (tỷ USD)',
        sector='Ngành'
    )
)
fig8.update_traces(textposition='top center')
fig8.update_layout(
    title='Tương quan ROE vs P/E theo Ngành',
    xaxis_title='P/E Trung vị (lần)',
    yaxis_title='ROE Trung vị (%)',
    template='plotly_white',
    showlegend=False
)
fig8.show()


In [17]:
max_roe_row = df_sec_lat.loc[df_sec_lat['median_roe_pct'].idxmax()]
max_mcap_row = df_sec_lat.loc[df_sec_lat['total_mcap_bn'].idxmax()]
mean_pe = float(df_sec_lat['median_pe'].mean())
high_roe_val = float(max_roe_row['median_roe_pct'])
high_roe_pe = float(max_roe_row['median_pe'])
high_roe_sector = max_roe_row['sector']
max_mcap_sector = max_mcap_row['sector']
max_mcap_val = float(max_mcap_row['total_mcap_bn'])
max_mcap_pe = float(max_mcap_row['median_pe'])

display(Markdown(f"""### Nhận định: Tương quan Hiệu quả Hoạt động (ROE) và Định giá (P/E)
- **Hiệu quả sinh lời ROE:** Ngành **{high_roe_sector}** ghi nhận tỷ suất sinh lời trên vốn chủ sở hữu ROE trung vị đạt **{high_roe_val:.1f}%**, đi cùng chỉ số định giá P/E ở mức **{high_roe_pe:.1f}x** (thấp hơn mức trung bình các ngành là **{mean_pe:.1f}x**).
- **Quy mô vốn hoá:** Ngành **{max_mcap_sector}** ghi nhận quy mô vốn hoá dẫn đầu đạt **{max_mcap_val:,.1f} tỷ USD**, với mức định giá P/E trung vị là **{max_mcap_pe:.1f}x**.
- **Đặc trưng phân bố:** Biểu đồ cho thấy sự phân hóa giữa các ngành, trong đó có ngành sinh lời ROE cao nhưng P/E thấp hơn mức trung bình các ngành, và có ngành quy mô vốn hoá lớn đi cùng với mức định giá P/E cao."""))


### Nhận định: Tương quan Hiệu quả Hoạt động (ROE) và Định giá (P/E)
- **Hiệu quả sinh lời ROE:** Ngành **Energy** ghi nhận tỷ suất sinh lời trên vốn chủ sở hữu ROE trung vị đạt **19.1%**, đi cùng chỉ số định giá P/E ở mức **15.0x** (thấp hơn mức trung bình các ngành là **19.8x**).
- **Quy mô vốn hoá:** Ngành **Technology** ghi nhận quy mô vốn hoá dẫn đầu đạt **29,822.1 tỷ USD**, với mức định giá P/E trung vị là **28.2x**.
- **Đặc trưng phân bố:** Biểu đồ cho thấy sự phân hóa giữa các ngành, trong đó có ngành sinh lời ROE cao nhưng P/E thấp hơn mức trung bình các ngành, và có ngành quy mô vốn hoá lớn đi cùng với mức định giá P/E cao.

## Hạn chế 

Một số chỉ số vĩ mô (cụ thể là `credit_spread_hy`) chỉ có dữ liệu từ ngày 2023-10-17 trong nguồn lakehouse, chưa bao quát toàn bộ chuỗi 10 năm như lãi suất trái phiếu và VIX. Chưa kiểm tra thiên lệch sống sót:** Phân tích chưa đánh giá ảnh hưởng của thiên lệch sống sót đối với các mã cổ phiếu bị hủy niêm yết trong quá khứ.
